This notebook implements logistic regression for sentiment analysis on tweets. Given a tweet, model will decide if it has a positive sentiment or a negative one. Unigrams are used as features
 
Functions:
* Extract features for logistic regression given some text
* Implement logistic regression from scratch
* Apply logistic regression on a natural language processing task
* Test logistic regression model
* Perform error analysis

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.2read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/twitter-tweets-sentiment-dataset/Tweets.csv


## Imports

In [2]:
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import TweetTokenizer
import re
import string

In [3]:
data = pd.read_csv("/kaggle/input/twitter-tweets-sentiment-dataset/Tweets.csv")
data.head()

,textID,text,selected_text,sentiment
0,cb774db0d1,"I`d have responded, if I were going","I`d have responded, if I were going",neutral
1,549e992a42,Sooo SAD I will miss you here in San Diego!!!,Sooo SAD,negative
2,088c60f138,my boss is bullying me...,bullying me,negative
3,9642c003ef,what interview! leave me alone,leave me alone,negative
4,358bd9e861,"Sons of ****, why couldn`t they put them on t...","Sons of ****,",negative


In [4]:
data=data.drop(columns=["textID","selected_text"])
data.head(5)

,text,sentiment
0,"I`d have responded, if I were going",neutral
1,Sooo SAD I will miss you here in San Diego!!!,negative
2,my boss is bullying me...,negative
3,what interview! leave me alone,negative
4,"Sons of ****, why couldn`t they put them on t...",negative


In [5]:
data.groupby(by="sentiment").count()

,text
sentiment,
negative,7781
neutral,11117
positive,8582


In [6]:
#Removing tweets with neutral sentiment
data = data[data["sentiment"] != "neutral"]

In [7]:
data.groupby(by="sentiment").count()

,text
sentiment,
negative,7781
positive,8582


In [8]:
data.sentiment = data["sentiment"].replace({"positive":1, "negative":0})
data.head(5)

/tmp/ipykernel_17/1518609471.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data.sentiment = data["sentiment"].replace({"positive":1, "negative":0})


,text,sentiment
1,Sooo SAD I will miss you here in San Diego!!!,0
2,my boss is bullying me...,0
3,what interview! leave me alone,0
4,"Sons of ****, why couldn`t they put them on t...",0
6,2am feedings for the baby are fun when he is a...,1


## Preprocessing

#### Process tweet
1. Changing the tweet to lower case
2. Removing punctuation
3. Removing stop words (and, the, is, etc.)
4. Stemming
5. Tokenizing

In [9]:
def process_tweets(tweet):
    '''
    tweet: a string containing the tweet
    tweet_clean : a list, contains preprocessed tweet as token
    '''
    stopwords_english = stopwords.words("english")
    keep_words = {"not", "no", "nor", "never", "n't", "but", "however", "very", "too", "so"}

    stopwords_english = set(stopwords_english) - keep_words
    stopwords_english = list(stopwords_english)
    
    stemmer = PorterStemmer()
    
    #Removing hashtags
    tweet = re.sub(r"#","",tweet)
    #Removing retweet symbol RT at start of the tweet + whitespaces after RT
    tweet = re.sub(r"^RT[\s]+", "", tweet)
    #Removing hyperlinks
    tweet = re.sub(r"https?:\/\/.*[\r\n]*", "", tweet)
    #Removing stock ticker symbol
    tweet = re.sub(r"\$\w+", "", tweet)

    #Tokenizer
    tokenizer = TweetTokenizer(preserve_case=False, strip_handles=True, reduce_len=True)
    tweet_token = tokenizer.tokenize(tweet)

    tweet_token = [tok for tok in tweet_token if not re.fullmatch(r"\W+",tok)]
    
    tweet_clean = []

    for word in tweet_token:
        if (word not in string.punctuation) and (word not in stopwords_english):
            tweet_clean.append(stemmer.stem(word))

    return tweet_clean  

In [10]:
def punctuation_count(tweet):
    '''
    tweet: string for which the punctuation cound needs to be calculated
    question_count: number of “?”
    exclam_count : number of “!”
    ellipsis_count: number of “...”
    h_smiley: number of “:)”
    s_smiley: number of “:(”
    '''

    question_count = exclam_count = ellipsis_count = h_smiley = s_smiley = 0

    tokenizer = TweetTokenizer(preserve_case=False, strip_handles=True, reduce_len=True)
    tweet_token = tokenizer.tokenize(tweet)
    tweet_token = [word for word in tweet_token if re.fullmatch(r"\W+",word)]

    for string in tweet_token:
        exclam_count += len(re.findall(r"\!",string))
        question_count += len(re.findall(r"\?",string))
        ellipsis_count += len(re.findall(r"\...",string))
        h_smiley += len(re.findall(r":\)",string))
        s_smiley += len(re.findall(r":\(",string))

    return question_count,exclam_count,ellipsis_count,h_smiley,s_smiley

#### Building word frequency
Building frquency dictionary for positive and negatgive word. 
{(word,1) : count, (word,0):count, ....}

In [11]:
def build_freqs(tweets, ys):
    '''
    input:
        tweets: a list of tweets
        ys: a (mx1) array containing positive (1) or negative label (0) for list of tweets
    output:
        freqs: a dictionary mapping each (word, sentiment) to its frequency
    '''

    yslist = np.squeeze(ys).tolist()

    freqs = {}
    
    for y, tweet in zip(yslist, tweets):
        for word in process_tweets(tweet):
            pair = (word, y)
            if (word,y) in freqs:
                freqs[pair] += 1
            else:
                freqs[pair] = 1

    return freqs

## Train and Test split

In [12]:
from sklearn.model_selection import train_test_split

train_x, test_x, train_y, test_y = train_test_split(data.text, data.sentiment,test_size = 0.2, random_state=42)

In [13]:
print("Train x shape: ", train_x.shape)
print("Train y shape: ", train_y.shape)
print("Test x shape: ", test_x.shape)
print("Test y shape: ", test_y.shape)

Train x shape:  (13090,)
Train y shape:  (13090,)
Test x shape:  (3273,)
Test y shape:  (3273,)


In [14]:
train_x = list(train_x)
train_y = np.array(train_y).reshape(-1,1)
test_x = list(test_x)
test_y = np.array(test_y).reshape(-1,1)

In [15]:
print("Train x shape: ", len(train_x))
print("Train y shape: ", train_y.shape)
print("Test x shape: ", len(test_x))
print("Test y shape: ", test_y.shape)

Train x shape:  13090
Train y shape:  (13090, 1)
Test x shape:  3273
Test y shape:  (3273, 1)


In [16]:
# Building word frequency dictionary for the train x corpus
freqs = build_freqs(train_x, train_y)

In [17]:
print("First 10 items in the Frequency dictionary")
count = 0
for key, value in freqs.items():
    print(key, freqs[key])
    count +=1
    if count == 10:
        break

First 10 items in the Frequency dictionary
('total', 0) 47
('bereft', 0) 1
('fault', 0) 12
('everi', 0) 29
('way', 0) 91
('_kate', 1) 1
('know', 1) 195
('heap', 1) 3
('awesom', 1) 223
('not', 1) 245


### Sentiment analysis using logistic regression

#### Sigmoid function

In [18]:
# sigmoid function
def sigmoid(z):
    s = 1/(1+np.exp(-z))
    return s

#### Cost function and Gradient descent

Cost : $$J = \frac{-1}{m} \times \left(\mathbf{y}^T \cdot log(\mathbf{h}) + \mathbf{(1-y)}^T \cdot log(\mathbf{1-h}) \right)$$ 
Gradient: The gradient of the cost function $J$ with respect to one of the weights $w_j$ is:

$$\mathbf{w} = \mathbf{w} - \frac{\alpha}{m} \times \left( \mathbf{x}^T \cdot \left( \mathbf{h-y} \right) \right)$$

In [19]:
def gradientdescent(x,y,w,alpha, num_iters):

    m = x.shape[0]
    for i in range(num_iters):
        z = np.dot(x, w)
        h = sigmoid(z)
        J = (-1/m) * ( np.dot(y.T, np.log(h)) + np.dot((1-y).T, np.log(1-h)) )
        dw = (1/m) * np.dot(x.T, h-y)
        w = w - (alpha * dw)    
    J = J.flatten()
    J = float(J[0])
    return J, w

#### Feature extraction

In [20]:
def extract_features(tweet, freqs, process_tweet=process_tweets):
    '''
    Input: 
        tweet: a string containing one tweet
        freqs: a dictionary corresponding to the frequencies of each tuple (word, label)
    Output: 
        x: a feature vector of dimension (1,8) => [1, freqofpositive, freqofnegative,
        question_count, exclam_count, ellipsis_count, h_smiley ,s_smiley ]
    '''
    word_l = process_tweet(tweet)
    
    x = np.zeros(8)
    x[0] = 1 
    
    for word in word_l:
        if (word,1) in freqs:
            x[1] += freqs[(word,1)]

        if (word,0) in freqs:
            x[2] += freqs[(word,0)]

    x[3],x[4],x[5],x[6],x[7]  = punctuation_count(tweet)
    
    x = x[None, :]  # adding batch dimension for further processing
    assert(x.shape == (1, 8))
    return x

In [21]:
# test on training data
tmp1 = extract_features(test_x[9], freqs)
print("Tweet:", test_x[9])
print("\nExtracted features:", tmp1)

Tweet: I`ve been losing myself into too many Taiwanese dramas.... ????, ??????, ????... this is not good. XD But Wu Chun! Eeeeeee.

Extracted features: [[1.000e+00 1.919e+03 1.718e+03 9.000e+00 1.000e+00 2.000e+00 0.000e+00
  0.000e+00]]


#### Model training

In [22]:
# X : features for all the tweets in train_x
X = np.zeros((len(train_x),8))

for i in range(len(train_x)):
    X[i,:] = extract_features(train_x[i], freqs)

# Y
Y = train_y

# Weight and Learning rate (alpha)
w = np.zeros((8,1))
alpha = 1e-9

In [23]:
J, w = gradientdescent(X, Y, w, alpha, num_iters=1500)

print(f"The cost after training is {J:.8f}.")
print(f"The resulting vector of weights is {w}")

The cost after training is 0.66568466.
The resulting vector of weights is [[ 1.37909762e-08]
 [ 1.85456541e-04]
 [-7.71191027e-05]
 [-1.11871788e-08]
 [ 1.19949369e-07]
 [-1.57853749e-08]
 [ 0.00000000e+00]
 [ 0.00000000e+00]]


#### Prediction model

In [24]:
# UNQ_C4 GRADED FUNCTION: predict_tweet
def predict_tweet(tweet, freqs, w):
    '''
    Input: 
        tweet: a string
        freqs: a dictionary corresponding to the frequencies of each tuple (word, label)
        w: (3,1) vector of weights
    Output: 
        y_pred: the probability of a tweet being positive or negative
    '''
    x = extract_features(tweet, freqs, process_tweet=process_tweets)
    z = np.dot(x, w)
    y_pred = sigmoid(z)
    
    return y_pred

In [25]:
def predict_model(x, freqs, w, predict_tweet=predict_tweet):

    y_hat = []

    for tweet in x:
        y_pred = predict_tweet(tweet, freqs, w)

        if y_pred > 0.5:
            y_hat.append(1.0)
        else:
            y_hat.append(0.0)

    y_hat = np.array(y_hat)
    
    return y_hat

In [26]:
train_pred = predict_model(train_x, freqs, w)
test_pred = predict_model(test_x, freqs, w)

In [27]:
train_accuracy = np.sum(train_y.reshape(-1) == train_pred)/len(train_y)
test_accuracy = np.sum(test_y.reshape(-1) == test_pred)/len(test_y)

print(f"Training accuracy =  {train_accuracy:.4f}")
print(f"Test accuracy = {test_accuracy:.4f}")

Training accuracy =  0.5837
Test accuracy = 0.5671


#### Error analysis

In [28]:
# Some error analysis done for you
for x,y in zip(test_x[0:10],test_y[0:10]):
    y_hat = predict_tweet(x, freqs, w)
    
    if np.abs(y - (y_hat > 0.5)) > 0:
        print('THE TWEET IS:', x)
        print('THE PROCESSED TWEET IS:', process_tweets(x))
        print('%d\t%0.8f\t' % (y, y_hat))

THE TWEET IS: Today is lame because I am not in Orlando  I am soooo looking forward to NEXT friday
THE PROCESSED TWEET IS: ['today', 'lame', 'not', 'orlando', 'sooo', 'look', 'forward', 'next', 'friday']
0	0.52388504	
THE TWEET IS: Gahh ! This weather sucksss !
THE PROCESSED TWEET IS: ['gahh', 'weather', 'sucksss']
0	0.50138136	
THE TWEET IS: Sitting in boring **** litterature listening to jack Johnson  missing the gf soooooooooooooooooooooooooooooooooooo much.
THE PROCESSED TWEET IS: ['sit', 'bore', 'litteratur', 'listen', 'jack', 'johnson', 'miss', 'gf', 'sooo', 'much']
0	0.50108556	
THE TWEET IS:  oh but that girl  but AIDAN!
THE PROCESSED TWEET IS: ['oh', 'but', 'girl', 'but', 'aidan']
0	0.52616956	
THE TWEET IS: I`ve been losing myself into too many Taiwanese dramas.... ????, ??????, ????... this is not good. XD But Wu Chun! Eeeeeee.
THE PROCESSED TWEET IS: ['lose', 'too', 'mani', 'taiwanes', 'drama', 'not', 'good', 'xd', 'but', 'wu', 'chun', 'eeee']
0	0.55561899	


/tmp/ipykernel_17/2390248778.py:8: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  print('%d\t%0.8f\t' % (y, y_hat))


1. The model is not performing well as unigrams are used and hence "not good" becomes "good" while tokenizing
2. feature representation is extremely compressed 
